## Generate GP curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from typing import Sequence

### Utility functions

In [ ]:
type ArrayLike = NDArray | Sequence[int | float]

def exp_kernel(A: NDArray, B: NDArray, ls: float) -> NDArray:
    # RBF kernel: k(xa, xb) = exp(-0.5 * ||xa - xb||^2) (lengthscale=1)
    A = np.asarray(A)  # (n_a, n_features)
    B = np.asarray(B)  # (n_b, n_features)

    # squared euclidean distance matrix (n_a, n_b)
    sqdist = np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=-1)

    return np.exp(-0.5 * sqdist / (ls**2))


def power_spectral_density(omega: NDArray, ls: float | NDArray, n_dims: int) -> NDArray:
    """
    Power spectral density for the ExpQuad kernel.

    .. math::

        S(\\boldsymbol\\omega) =
            (\\sqrt(2 \\pi)^D \\prod_{i}^{D}\\ell_i
            \\exp\\left( -\\frac{1}{2} \\sum_{i}^{D}\\ell_i^2 \\omega_i^{2} \\right)

    Args:
        omega: array of shape (m_star, d), frequencies at which to evaluate the PSD.
        ls:    lengthscale(s), either a scalar or array of shape (d,).
        n_dims: number of input dimensions D.

    Returns:
        Array of shape (m_star,), one PSD value per basis function.
    """
    ls_arr = np.ones(n_dims) * ls          # (d,)
    c = np.power(np.sqrt(2.0 * np.pi), n_dims)
    exp = np.exp(-0.5 * np.dot(np.square(omega), np.square(ls_arr)))  # (m_star,)
    return c * np.prod(ls_arr) * exp       # (m_star,)

def calc_eigenvalues(L: ArrayLike, m: int, d: int) -> NDArray:
    """
    Calculate eigenvalues of the Laplacian on [-L1,L1] x ... x [-Ld,Ld]
    with Dirichlet boundary conditions, returning the m smallest.

    Parameters
    ----------
    L : NDArray, shape (d,)
        Domain half-widths per dimension.
    m : int
        Number of eigenfunctions to return.
    d : int
        Number of input dimensions.

    Returns
    -------
    selected_per_dim_eigenvalues : NDArray, shape (m,d)
        The m smallest eigenvalues, sorted ascending.
    """
    L = np.asarray(L, dtype=float)
    
    # Number of indices per dimension, scaled by relative domain size
    N_per_dim = np.ceil(m ** (1 / d) * L / L.min()).astype(int)

    # Build full multi-index grid (Cartesian product of per-dim indices)
    temp = [np.arange(1, 1 + N_per_dim[dim]) for dim in range(d)]
    grids = np.meshgrid(*temp, indexing='ij')
    NN = np.vstack([g.ravel() for g in grids]).T      # (N_total, d)

    # Compute all eigenvalues
    per_dim_eigvals = np.square((np.pi * NN) / (2 * L)) # (N_total, d)
    all_eigenvalues = np.sum(per_dim_eigvals, axis=1)

    # Sort and keep the m smallest
    sort_idx = np.argsort(all_eigenvalues)[:m]
    selected_per_dim_eigenvalues = per_dim_eigvals[sort_idx]
    # NN = NN[sort_idx]

    return selected_per_dim_eigenvalues

def calc_eigenvectors(Xs: NDArray, L: ArrayLike, per_dim_eigvals: NDArray) -> NDArray:
    """Calculate eigenvectors of the Laplacian.
    These are used as basis vectors in the HSGP approximation.

    Parameters
    ----------
    Xs              : NDArray, shape (n_samples, d)
    L               : NDArray, shape (d,)
    per_dim_eigvals : NDArray, shape (m, d)
        Per dimension eigenvalues, with the smallest sum

    Returns
    -------
    phi : NDArray, shape (n_samples, m)
    """
    L = np.asarray(L, dtype=float)
    
    # (1, m, d) * (n_samples, 1, d) -> (n_samples, m, d)
    term1 = np.sqrt(per_dim_eigvals)[None, :, :]          # (1, m, d)
    term2 = Xs[:, None, :] + L[None, None, :]     # (n_samples, 1, d) + (1, 1, d)
    c = 1.0 / np.sqrt(L)                          # (d,)

    phi = c * np.sin(term1 * term2)               # (n_samples, m, d)
    return np.prod(phi, axis=-1)                  # (n_samples, m)

### Initialize parameters

In [ ]:
ls = 0.3
var_f = 1.0
var_n = 0.01
N_train = 50
N_plot = 200
n_dim  = 1

L = [1]
m = 50
margin = 1.1

### Generate measurements from a 1D GP

$$
y_i = f(x_i) + \varepsilon_i \quad  \varepsilon_i \sim \mathcal{N}(0, \sigma_n^2)
$$

Where 
$$
f \sim \mathcal{GP}(0, k(x,x'))
$$

using the Squared Exponential kernel $k(x,x') = \sigma_f^2 e^ {-\frac{||x - x'||^2}{2 \ell^2}}$

In [ ]:
# Generate the test_test_inputs X*
x_train = np.linspace(-L[0],L[0],N_train).reshape(-1, 1) # (n_samples, d)
x_plot = np.linspace(-2,2,N_plot).reshape(-1, 1) # (n_samples, d)

# Sample latent function f jointly at both plot and training points
x_all = np.vstack([x_plot, x_train])
K_all = var_f * exp_kernel(x_all, x_all, ls)

# Sample the smooth latent function
rng = np.random.default_rng(seed=5)
f_all = rng.multivariate_normal(np.zeros(len(x_all)), K_all)

# Split into plot and train portions
f_plot = f_all[:N_plot]
f_at_train = f_all[N_plot:]

# Add noise to get observations y
y_train = f_at_train + rng.normal(0, np.sqrt(var_n), size=N_train)

# Compute standard deviation for the confidence band
K_ff = var_f*exp_kernel(x_plot, x_plot, ls)
std_f = np.sqrt(np.diag(K_ff))

fig, axs = plt.subplots(1, 1)
axs.fill_between(x_plot.flatten(), -2*std_f, 2*std_f, alpha=0.3, color='gray', label='±2σ of f')
axs.plot(x_plot.flatten(), f_plot)
axs.scatter(x_train.flatten(), y_train, c='r')
axs.legend()
plt.show()

### Compare performance of exact GP and HSGP

In [ ]:
# Compute posterior from exact GP
x_test = x_plot

K_yy = var_f * exp_kernel(x_train, x_train, ls) + np.eye(N_train) * var_n
K_yy_inv = np.linalg.solve(K_yy, np.eye(N_train))

K_sx = var_f * exp_kernel(x_test, x_train, ls)
K_xs = var_f * exp_kernel(x_train, x_test, ls)
K_ss = var_f * exp_kernel(x_test, x_test, ls)

gp_cov = K_ss - K_sx @ K_yy_inv @ K_xs
gp_mean = K_sx @ K_yy_inv @ y_train


$$
\hat{f_*} = \Phi _* (\sigma_n^2 \Lambda^{-1} + \Phi^\top \Phi)^{-1} y
$$

$$
cov(f_*) = \sigma_n^2\Phi _* (\sigma_n^2 \Lambda^{-1} + \Phi^\top \Phi)^{-1} \Phi^\top_*
$$

In [ ]:

# Compute posterior of HSGP
eigvals = calc_eigenvalues(L, m, n_dim)  # type: ignore # (m_start, d)
phi = calc_eigenvectors(x_train, L, eigvals)  # (n_samples, m_star)
phi_star = calc_eigenvectors(x_test, L, eigvals)
omega = np.sqrt(eigvals)  # (m_star, d)
psd = power_spectral_density(omega, ls, n_dim)

Lambda = np.diag(psd)                                        # (m_star, m_star)
A = var_n * np.linalg.inv(Lambda) + phi.T @ phi         # (m_star, m_star)
alpha = np.linalg.solve(A, phi.T @ y_train)                # (m_star,)
hsgp_mean = phi_star @ alpha                                # (n_test,)

# Predictive covariance
A_inv = np.linalg.inv(A)
hsgp_cov = var_n * phi_star @ A_inv @ phi_star.T        # (n_test, n_test)  


In [ ]:

gp_std = np.diag(gp_cov)
hsgp_std = np.diag(hsgp_cov)
print(hsgp_std)             

fig, axs = plt.subplots(1, 1)
# axs.plot(x_plot.flatten(), f_plot)
axs.plot(x_test.flatten(), gp_mean)
axs.plot(x_test.flatten(), hsgp_mean)
axs.scatter(x_train.flatten(), y_train, c='r', s=0.5)
axs.fill_between(x_test.flatten(),gp_mean -2*gp_std, gp_mean + 2*gp_std, alpha=0.3, color='gray', label='±2σ')
axs.fill_between(x_test.flatten(),hsgp_mean -20*hsgp_std, hsgp_mean + 20*hsgp_std, alpha=0.3, color='blue', label='±2σ')
plt.show()